# BLOQUE 1: IMPORTACIONES Y CONFIGURACIÓN INICIAL

Importa todas las librerías necesarias para el pipeline de Marketing Mix Modeling (MMM) y configura las advertencias.

In [33]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import pyodbc
import pywt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential

warnings.filterwarnings('ignore')

# BLOQUE 2: FUNCIÓN DE DENOISING WAVELET

Limpia el ruido de los datos usando transformada wavelet para que el LSTM pueda ver tendencias reales sin distraerse con picos sin sentido.

In [ ]:
def aplicar_wavelet_denoising(serie, wavelet='db4'):
    valores = serie.astype(float).values
    if len(valores) < 4:
        return valores
    coefs = pywt.wavedec(valores, wavelet, mode='per')
    mad = np.median(np.abs(coefs[-1] - np.median(coefs[-1])))
    sigma = mad / 0.6745 if mad > 0 else 0.1
    umbral = sigma * np.sqrt(2 * np.log(len(valores)))
    coefs[1:] = [pywt.threshold(c, value=umbral, mode='soft') for c in coefs[1:]]
    return pywt.waverec(coefs, wavelet, mode='per')[: len(valores)]

# BLOQUE 3: FUNCIÓN PARA CONSTRUIR Y ENTRENAR LSTM

Construye y entrena una red neuronal LSTM con dos capas para predecir matrículas a partir de inversión, leads y meta de estudiantes.

In [34]:
def construir_y_entrenar_lstm(X_scaled, y_scaled):
    X_3D = np.reshape(X_scaled, (X_scaled.shape[0], 1, X_scaled.shape[1]))
    modelo = Sequential([
        LSTM(64, activation='tanh', return_sequences=True, input_shape=(X_3D.shape[1], X_3D.shape[2])),
        Dropout(0.1),
        LSTM(32, activation='tanh', return_sequences=False),
        Dense(16, activation='relu'),
        Dense(1, activation='linear'),
    ])
    modelo.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.002), loss=tf.keras.losses.Huber())
    modelo.fit(X_3D, y_scaled, epochs=40, batch_size=4, verbose=0)
    return modelo.predict(X_3D, verbose=0)

# BLOQUE 4: NORMALIZACIÓN Y CLASIFICACIÓN

- `normalizar_texto()`: Limpia y estandariza texto (mayúsculas, sin acentos, sin espacios múltiples)
- `clasificar_fuente()`: Clasifica campañas como GADS, META u ORGANICO según palabras clave

In [35]:
def normalizar_texto(texto):
    if pd.isna(texto):
        return ''
    texto = str(texto).strip().upper()
    replacements = {'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U', 'Ñ': 'N'}
    for k, v in replacements.items():
        texto = texto.replace(k, v)
    return re.sub(r'\s+', ' ', texto)

def clasificar_fuente(campana):
    campana_upper = str(campana).upper()
    if 'GADS' in campana_upper or 'GOOGLE' in campana_upper:
        return 'GADS'
    elif 'META' in campana_upper or 'FACEBOOK' in campana_upper or 'INSTAGRAM' in campana_upper:
        return 'META'
    else:
        return 'ORGANICO'

# BLOQUE 5: PIPELINE PRINCIPAL - CONFIGURACIÓN Y CONEXIÓN SQL

Configura la conexión a SQL Server, define la función principal del pipeline y muestra mensaje de inicio.

In [50]:
print("Probando conexión a SQL Server...")
try:
    import pyodbc
    conn = pyodbc.connect(
        'DRIVER={ODBC Driver 17 for SQL Server};'
        'SERVER=172.16.1.33;'
        'DATABASE=CUN_REPOSITORIO;'
        'UID=coe;'
        'PWD=C1n1283*uys;',
        timeout=5
    )
    print("✅ CONEXIÓN EXITOSA")
    conn.close()
except Exception as e:
    print(f"❌ ERROR: {e}")
    print("   Posibles causas:")
    print("   1. Servidor caído o inaccesible")
    print("   2. Credenciales incorrectas")
    print("   3. Driver ODBC no instalado")
    print("   4. Firewall bloqueando la conexión")

Probando conexión a SQL Server...
✅ CONEXIÓN EXITOSA


In [52]:
def ejecutar_pipeline_mmm_produccion(presupuesto_objetivo=7000000000):
    
    print("=" * 80)
    print("🚀 INICIANDO PIPELINE MMM")
    print("=" * 80)
    
    print("\n[PASO 1] Obteniendo directorio de trabajo...")
    directorio_trabajo = os.getcwd()  # <--- CAMBIADO: __file__ no existe en Jupyter
    print(f"        ✅ Directorio: {directorio_trabajo}")
    
    print("\n[PASO 2] Configurando conexión a SQL Server...")
    string_conexion = (
        'DRIVER={ODBC Driver 17 for SQL Server};'
        'SERVER=172.16.1.33;'
        'DATABASE=CUN_REPOSITORIO;'
        'UID=coe;'
        'PWD=C1n1283*uys;'
    )
    print("        ✅ Conexión configurada")
    
    print("\n[PASO 3] Conectando a SQL Server...")
    try:
        conn = pyodbc.connect(string_conexion, timeout=20)
        print("        ✅ CONEXIÓN EXITOSA A SQL SERVER")
    except Exception as e:
        print(f"        ❌ ERROR DE CONEXIÓN: {e}")
        return None, None, None, None, None, None
    
    print("\n[PASO 4] Ejecutando consulta: METAS...")
    try:
        query_metas = """
            SELECT m.PROGRAMA_ACADEMICO, m.MODALIDAD, m.PERIODO, SUM(m.META_EN_ESTUDIANTES) AS META
            FROM financiera.metas m
            INNER JOIN CUN_REPOSITORIO.dbo.Periodos_Calendario p ON m.PERIODO = p.cod_periodo
            WHERE m.FUERZA_COMERCIAL = 'CONTACT'
              AND p.fec_inicio >= '2026-01-01'
            GROUP BY m.PROGRAMA_ACADEMICO, m.MODALIDAD, m.PERIODO;
        """
        df_metas = pd.read_sql(query_metas, conn)
        print(f"        ✅ METAS CARGADAS: {len(df_metas)} registros")
    except Exception as e:
        print(f"        ❌ ERROR EN METAS: {e}")
        return None, None, None, None, None, None
    
    print("\n[PASO 5] Normalizando textos de METAS...")
    try:
        df_metas['PROGRAMA_ACADEMICO'] = df_metas['PROGRAMA_ACADEMICO'].apply(normalizar_texto)
        df_metas['PERIODO'] = df_metas['PERIODO'].apply(normalizar_texto)
        df_metas['MODALIDAD'] = df_metas['MODALIDAD'].apply(normalizar_texto)
        print("        ✅ TEXTOS NORMALIZADOS")
    except Exception as e:
        print(f"        ❌ ERROR EN NORMALIZACIÓN: {e}")
        return None, None, None, None, None, None
    
    print("\n[PASO 6] Generando listas autorizadas...")
    try:
        periodos_autorizados = df_metas['PERIODO'].unique().tolist()
        programas_autorizados = df_metas['PROGRAMA_ACADEMICO'].unique().tolist()
        str_periodos = ','.join([f"'{p}'" for p in periodos_autorizados])
        print(f"        ✅ {len(periodos_autorizados)} períodos autorizados")
        print(f"        ✅ {len(programas_autorizados)} programas autorizados")
        print(f"        ✅ Primeros 3 períodos: {periodos_autorizados[:3]}")
    except Exception as e:
        print(f"        ❌ ERROR EN LISTAS: {e}")
        return None, None, None, None, None, None
    
    print("\n[PASO 7] Ejecutando consulta: VENTAS...")
    try:
        query_ventas = f"""
            SELECT convertido, nombre_de_campaña_mercadeo, id_base, periodo, 
                   programalimpio AS PROGRAMA, canal_fuente, modalidad AS MODALIDAD
            FROM crm.Registros_CRM 
            WHERE fuerzacomercial = 'Contact'
              AND convertido='VENTA' 
              AND periodo IN ({str_periodos});
        """
        df_ventas = pd.read_sql(query_ventas, conn)
        print(f"        ✅ VENTAS CARGADAS: {len(df_ventas)} registros")
    except Exception as e:
        print(f"        ❌ ERROR EN VENTAS: {e}")
        return None, None, None, None, None, None
    
    print("\n[PASO 8] Ejecutando consulta: OPORTUNIDADES...")
    try:
        query_oportunidades = f"""
            SELECT convertido, nombre_de_campaña_mercadeo, id_base, periodo, 
                   programalimpio AS PROGRAMA, canal_fuente
            FROM crm.Registros_CRM 
            WHERE fuerzacomercial = 'Contact' 
              AND creador_lead = 'MARKETING'
              AND tipo_registro IN ('POSIBLE_CLIENTE', 'OPORTUNIDAD')
              AND periodo IN ({str_periodos});
        """
        df_oportunidades = pd.read_sql(query_oportunidades, conn)
        print(f"        ✅ OPORTUNIDADES CARGADAS: {len(df_oportunidades)} registros")
    except Exception as e:
        print(f"        ❌ ERROR EN OPORTUNIDADES: {e}")
        return None, None, None, None, None, None
    
    print("\n[PASO 9] Ejecutando consulta: GASTOS...")
    try:
        query_gastos = (
            'SELECT * FROM [CUN_REPOSITORIO].[FINANCIERA].[Leads_campaña_gasto]'
            ' WHERE Año=2026;'
        )
        df_gastos = pd.read_sql(query_gastos, conn)
        print(f"        ✅ GASTOS CARGADOS: {len(df_gastos)} registros")
        print(f"        ✅ Columnas de gastos: {list(df_gastos.columns)}")
    except Exception as e:
        print(f"        ❌ ERROR EN GASTOS: {e}")
        return None, None, None, None, None, None
    
    print("\n[PASO 10] Cerrando conexión...")
    try:
        conn.close()
        print("        ✅ CONEXIÓN CERRADA")
    except Exception as e:
        print(f"        ❌ ERROR AL CERRAR: {e}")
    
    print("\n" + "=" * 80)
    print("✅ CARGA DE DATOS COMPLETADA CON ÉXITO")
    print("=" * 80)
    print(f"   📊 df_metas: {len(df_metas)} registros")
    print(f"   📊 df_ventas: {len(df_ventas)} registros")
    print(f"   📊 df_oportunidades: {len(df_oportunidades)} registros")
    print(f"   📊 df_gastos: {len(df_gastos)} registros")
    print("=" * 80)
    print("✅ LISTO PARA EL BLOQUE 7")
    print("=" * 80)
    
    return df_metas, df_ventas, df_oportunidades, df_gastos, periodos_autorizados, programas_autorizados


# =====================================================================
# EJECUTAR LA FUNCIÓN
# =====================================================================
print("\n" + "=" * 80)
print("🔥 EJECUTANDO PIPELINE MMM")
print("=" * 80 + "\n")

df_metas, df_ventas, df_oportunidades, df_gastos, periodos_autorizados, programas_autorizados = ejecutar_pipeline_mmm_produccion(presupuesto_objetivo=7000000000)

if df_metas is not None:
    print("\n✅ PIPELINE COMPLETADO. Variables disponibles:")
    print(f"   df_metas: {len(df_metas)} registros")
    print(f"   df_ventas: {len(df_ventas)} registros")
    print(f"   df_oportunidades: {len(df_oportunidades)} registros")
    print(f"   df_gastos: {len(df_gastos)} registros")
    print(f"   periodos_autorizados: {len(periodos_autorizados)} períodos")
    print(f"   programas_autorizados: {len(programas_autorizados)} programas")
else:
    print("\n❌ PIPELINE FALLÓ. Revise los errores arriba.")


🔥 EJECUTANDO PIPELINE MMM

🚀 INICIANDO PIPELINE MMM

[PASO 1] Obteniendo directorio de trabajo...
        ✅ Directorio: c:\Users\juan_garnicac\Documents\ProyectosVisual\Meridian\presentacion

[PASO 2] Configurando conexión a SQL Server...
        ✅ Conexión configurada

[PASO 3] Conectando a SQL Server...
        ✅ CONEXIÓN EXITOSA A SQL SERVER

[PASO 4] Ejecutando consulta: METAS...
        ✅ METAS CARGADAS: 317 registros

[PASO 5] Normalizando textos de METAS...
        ✅ TEXTOS NORMALIZADOS

[PASO 6] Generando listas autorizadas...
        ✅ 16 períodos autorizados
        ✅ 56 programas autorizados
        ✅ Primeros 3 períodos: ['2026A', '2026B', '2026C']

[PASO 7] Ejecutando consulta: VENTAS...
        ✅ VENTAS CARGADAS: 27120 registros

[PASO 8] Ejecutando consulta: OPORTUNIDADES...
        ✅ OPORTUNIDADES CARGADAS: 416985 registros

[PASO 9] Ejecutando consulta: GASTOS...
        ✅ GASTOS CARGADOS: 30845 registros
        ✅ Columnas de gastos: ['Periodo', 'Campaña', 'Año', 'Me

In [53]:
# =====================================================================
# BLOQUE 7: DETECCIÓN DE COLUMNAS Y LIMPIEZA DE DATOS
# =====================================================================

print("\n" + "=" * 80)
print("🔧 BLOQUE 7: DETECCIÓN DE COLUMNAS Y LIMPIEZA")
print("=" * 80)

# 1. DETECCIÓN AUTOMÁTICA DE COLUMNAS EN DATAFRAME DE GASTOS
print("\n[1/4] Detectando columnas en df_gastos...")
gasto_col = next(c for c in df_gastos.columns if any(k in c.upper() for k in ['GASTO', 'INVERSION']))
campana_gasto_col = next(c for c in df_gastos.columns if any(k in c.upper() for k in ['CAMPA']))
periodo_gasto_col = next(c for c in df_gastos.columns if any(k in c.upper() for k in ['PERIODO', 'AÑO', 'ANO']))
prog_gasto_col = next(c for c in df_gastos.columns if 'PROGRAMA' in c.upper())

print(f"   ✅ gasto_col: {gasto_col}")
print(f"   ✅ campana_gasto_col: {campana_gasto_col}")
print(f"   ✅ periodo_gasto_col: {periodo_gasto_col}")
print(f"   ✅ prog_gasto_col: {prog_gasto_col}")

# 2. NORMALIZACIÓN DE TEXTOS EN TODOS LOS DATAFRAMES
print("\n[2/4] Normalizando textos...")
df_oportunidades['PROGRAMA'] = df_oportunidades['PROGRAMA'].apply(normalizar_texto)
df_ventas['PROGRAMA'] = df_ventas['PROGRAMA'].apply(normalizar_texto)
df_gastos[prog_gasto_col] = df_gastos[prog_gasto_col].apply(normalizar_texto)

df_oportunidades['periodo'] = df_oportunidades['periodo'].apply(normalizar_texto)
df_ventas['periodo'] = df_ventas['periodo'].apply(normalizar_texto)
df_gastos[periodo_gasto_col] = df_gastos[periodo_gasto_col].apply(normalizar_texto)
print("   ✅ Textos normalizados")

# 3. FILTRADO: SOLO PROGRAMAS Y PERÍODOS AUTORIZADOS
print("\n[3/4] Filtrando programas y períodos autorizados...")
df_oportunidades = df_oportunidades[df_oportunidades['PROGRAMA'].isin(programas_autorizados)]
df_ventas = df_ventas[df_ventas['PROGRAMA'].isin(programas_autorizados)]
df_gastos = df_gastos[df_gastos[prog_gasto_col].isin(programas_autorizados)]
print(f"   ✅ Oportunidades filtradas: {len(df_oportunidades)}")
print(f"   ✅ Ventas filtradas: {len(df_ventas)}")
print(f"   ✅ Gastos filtrados: {len(df_gastos)}")

# 4. CLASIFICACIÓN DE FUENTES DE CAMPAÑAS
print("\n[4/4] Clasificando fuentes de campañas...")
df_oportunidades['Fuente Clasificada'] = df_oportunidades['nombre_de_campaña_mercadeo'].apply(clasificar_fuente)
df_ventas['Fuente Clasificada'] = df_ventas['nombre_de_campaña_mercadeo'].apply(clasificar_fuente)
df_gastos['Fuente Clasificada'] = df_gastos[campana_gasto_col].apply(clasificar_fuente)
print("   ✅ Fuentes clasificadas")

print("\n" + "=" * 80)
print("✅ BLOQUE 7 COMPLETADO")
print("=" * 80)


🔧 BLOQUE 7: DETECCIÓN DE COLUMNAS Y LIMPIEZA

[1/4] Detectando columnas en df_gastos...
   ✅ gasto_col: Gasto_Distribuido
   ✅ campana_gasto_col: Campaña
   ✅ periodo_gasto_col: Periodo
   ✅ prog_gasto_col: PROGRAMA

[2/4] Normalizando textos...
   ✅ Textos normalizados

[3/4] Filtrando programas y períodos autorizados...
   ✅ Oportunidades filtradas: 386926
   ✅ Ventas filtradas: 27017
   ✅ Gastos filtrados: 27405

[4/4] Clasificando fuentes de campañas...
   ✅ Fuentes clasificadas

✅ BLOQUE 7 COMPLETADO


# BLOQUE 7: DETECCIÓN DE COLUMNAS Y LIMPIEZA DE DATOS


In [54]:
# =====================================================================
# BLOQUE 8: CONSTRUCCIÓN DE BASE DE DATOS MAESTRA
# =====================================================================

print("\n" + "=" * 80)
print("🔧 BLOQUE 8: CONSTRUCCIÓN BASE DE DATOS MAESTRA")
print("=" * 80)

# 1. DICCIONARIO DE METAS POR PROGRAMA Y PERIODO
print("\n[1/5] Creando diccionario de metas...")
dict_metas_est = df_metas.groupby(['PROGRAMA_ACADEMICO', 'PERIODO'])['META'].sum().to_dict()
print(f"   ✅ {len(dict_metas_est)} combinaciones (programa, periodo)")

# 2. LISTA PARA ALMACENAR REGISTROS
registros_maestros = []
unique_combinations = df_metas.set_index(['PROGRAMA_ACADEMICO', 'MODALIDAD', 'PERIODO']).index.unique()
print(f"   ✅ {len(unique_combinations)} combinaciones únicas (programa, modalidad, periodo)")

# 3. RECORRE CADA COMBINACIÓN
print("\n[2/5] Procesando combinaciones...")
for prog, mod, per in unique_combinations:
    meta_programa_val = dict_metas_est.get((prog, per), 15)
    gasto_sub = df_gastos[(df_gastos[prog_gasto_col] == prog) & (df_gastos[periodo_gasto_col] == per)]
    
    # ASIGNACIÓN DE CLUSTER POR FACULTAD
    if any(k in prog for k in ['ADMINISTRACION', 'CONTADURIA', 'NEGOCIOS', 'MERCADEO', 'EMPRESAS', 'PUBLICIDAD']):
        cluster_facultad = 'CIENCIAS ECONOMICAS Y ADMINISTRATIVAS'
    elif any(k in prog for k in ['SISTEMAS', 'INDUSTRIAL', 'INGENIERIA', 'TECNOLOGIA']):
        cluster_facultad = 'INGENIERIA Y TECNOLOGIA'
    elif any(k in prog for k in ['COMUNICACION', 'DISEÑO', 'AUDIOVISUALES', 'MEDIOS']):
        cluster_facultad = 'COMUNICACION Y DISEÑO'
    else:
        cluster_facultad = 'OTRAS DISCIPLINAS / INSTITUCIONAL'
    
    if not gasto_sub.empty:
        for _, row_g in gasto_sub.iterrows():
            camp = row_g[campana_gasto_col]
            fuente = row_g['Fuente Clasificada']
            inv_total_campana = float(row_g[gasto_col])
            
            # CONTEO DE OPORTUNIDADES Y VENTAS
            l_cnt = len(df_oportunidades[(df_oportunidades['PROGRAMA'] == prog) & (df_oportunidades['periodo'] == per) & (df_oportunidades['nombre_de_campaña_mercadeo'] == camp)])
            v_cnt = len(df_ventas[(df_ventas['PROGRAMA'] == prog) & (df_ventas['periodo'] == per) & (df_ventas['nombre_de_campaña_mercadeo'] == camp)])
            total_leads_campana_global = len(df_oportunidades[(df_oportunidades['periodo'] == per) & (df_oportunidades['nombre_de_campaña_mercadeo'] == camp)])
            
            # PRORRATEO DE INVERSIÓN
            if total_leads_campana_global > 0 and l_cnt > 0:
                inv_distribuida = inv_total_campana * (l_cnt / total_leads_campana_global)
            else:
                progs_count = df_gastos[(df_gastos[campana_gasto_col] == camp) & (df_gastos[periodo_gasto_col] == per)][prog_gasto_col].nunique()
                inv_distribuida = inv_total_campana / progs_count if progs_count > 0 else inv_total_campana
            
            registros_maestros.append({
                'Periodo Meta': per,
                'Cluster Facultad': cluster_facultad,
                'Programa Academico': prog,
                'Modalidad': mod,
                'Campana Mercadeo': camp,
                'Fuente Clasificada': fuente,
                'Oportunidades Totales (Leads)': l_cnt,
                'Matriculas Reales': v_cnt,
                'Meta Estudiantes': meta_programa_val,
                'Inversion Gasto Distribuido': inv_distribuida,
            })

print(f"   ✅ {len(registros_maestros)} registros maestros creados")

# 4. CREACIÓN DE DATAFRAME BASE
print("\n[3/5] Creando DataFrame base...")
df_base_campanas = pd.DataFrame(registros_maestros).drop_duplicates(subset=['Periodo Meta', 'Programa Academico', 'Modalidad', 'Campana Mercadeo'])
df_base_campanas['Inversion Gasto Distribuido'] = df_base_campanas['Inversion Gasto Distribuido'].fillna(0.0)
print(f"   ✅ DataFrame creado: {len(df_base_campanas)} registros")

print("\n" + "=" * 80)
print("✅ BLOQUE 8 COMPLETADO")
print("=" * 80)


🔧 BLOQUE 8: CONSTRUCCIÓN BASE DE DATOS MAESTRA

[1/5] Creando diccionario de metas...
   ✅ 317 combinaciones (programa, periodo)
   ✅ 317 combinaciones únicas (programa, modalidad, periodo)

[2/5] Procesando combinaciones...
   ✅ 20612 registros maestros creados

[3/5] Creando DataFrame base...
   ✅ DataFrame creado: 2197 registros

✅ BLOQUE 8 COMPLETADO
